In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import QiskitRuntimeService


# needed for running quantum programs on IBM backends
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator
service = QiskitRuntimeService()
from qiskit import transpile
import json

In [ ]:
def ideal_sim():
    all_ideal_results = [] 
    ideal_simulator = AerSimulator()
    ideal_sampler = Sampler(mode=ideal_simulator)

    # Create the quantum circuit
    for n in range(2, 21):
        ghz_circuit = QuantumCircuit(n)

        ghz_circuit.h(0)

        # Entangle the remaining qubits with the first qubit
        for qubit in range(1, n):
            ghz_circuit.cx(0, qubit)

        # Measure all qubits
        ghz_circuit.measure_all()
        print(ghz_circuit)

        # Create a sampler that uses the ideal simulator

        # Number of circuit executions
        shots = 15000

        # Run the GHZ circuit
        ideal_job = ideal_sampler.run(
            [ghz_circuit],
            shots=shots
        )

        # Retrieve the results
        ideal_result = ideal_job.result()
        ideal_counts = ideal_result[0].data.meas.get_counts()

        all_ideal_results.append(ideal_counts)

    return all_ideal_results

In [ ]:
ideal_results = ideal_sim()
print(ideal_results)

with open("ideal_output.json", "w") as file:
    json.dump(ideal_results, file)

In [ ]:
backend = service.backend('ibm_fez')

print(f"Source backend: {backend.name}")


In [ ]:
def noisy_sim():
    all_noisy_results = []
    noisy_simulator = AerSimulator.from_backend(backend)
    print(f"Noisy simulator: {noisy_simulator.name}")
    noisy_sampler = Sampler(mode=noisy_simulator)

    # Create the quantum circuit
    for n in range(2, 21):
        ghz_circuit = QuantumCircuit(n)
        ghz_circuit.h(0)

        # Entangle the remaining qubits with the first qubit
        for qubit in range(1, n):
            ghz_circuit.cx(0, qubit)

        # Measure all qubits
        ghz_circuit.measure_all()
        print(ghz_circuit)
        noisy_compiled_circuit = transpile(
            ghz_circuit,
            backend=noisy_simulator,
            optimization_level=1
        )
        # Create a sampler that uses the noisy simulator

        # Number of circuit executions
        shots = 15000

        # Run the GHZ circuit
        noisy_job = noisy_sampler.run(
            [noisy_compiled_circuit],
            shots=shots
        )

        # Retrieve the results
        noisy_result = noisy_job.result()
        noisy_counts = noisy_result[0].data.meas.get_counts()

        all_noisy_results.append(noisy_counts)

    return all_noisy_results

In [ ]:
noisy_results = noisy_sim()
print(noisy_results)

with open("noisy_output.json", "w") as file:
    json.dump(noisy_results, file)

In [ ]:
def fidelity(results):
    fidelity_values = []
    i = 2
    for run in results:
        valid = 0
        invalid = 0

        for key, value in run.items():
            if len(set(key)) == 1:
                valid += value
            else:
                invalid += value


        temp = []
        calc_fidelity = (1 - invalid/(valid+invalid))
        temp.extend([valid, invalid, calc_fidelity])
        fidelity_values.append(temp)
        print(f"({i},{calc_fidelity})")
        i += 1
    return fidelity_values

In [ ]:
noisy_fidelity = fidelity(noisy_results)

In [ ]:
ideal_f = fidelity(ideal_results)

In [ ]:
from collections import defaultdict
from qiskit import transpile
from qiskit.circuit import QuantumCircuit


def print_circuit_metrics(circuit: QuantumCircuit, backend, transpile_kwargs: dict = None):

    transpile_kwargs = transpile_kwargs or {}

    routing_pass_names = {
        "SabreSwap", "StochasticSwap", "LookaheadSwap", "BasicSwap",
    }
    routing_swap_count = {"value": None}

    def _capture_routing_swaps(**kwargs):
        pass_ = kwargs.get("pass_")
        dag = kwargs.get("dag")
        if pass_ is not None and pass_.__class__.__name__ in routing_pass_names:
            swaps_here = dag.count_ops().get("swap", 0)
            routing_swap_count["value"] = swaps_here

    user_callback = transpile_kwargs.pop("callback", None)

    def _combined_callback(**kwargs):
        _capture_routing_swaps(**kwargs)
        if user_callback:
            user_callback(**kwargs)

    compiled = transpile(
        circuit, backend=backend, callback=_combined_callback, **transpile_kwargs
    )

    depth = compiled.depth()


    counts_by_arity = defaultdict(lambda: defaultdict(int))
    swap_count = 0
    two_qubit_ops = 0

    for instruction in compiled.data:
        op = instruction.operation
        qubits = instruction.qubits
        n_qubits = len(qubits)


        if op.name in ("barrier",):
            continue

        counts_by_arity[n_qubits][op.name] += 1

        if op.name == "swap":
            swap_count += 1
        if n_qubits == 2:
            two_qubit_ops += 1

    # --- Backend name (works for both BackendV1 and BackendV2) ---
    backend_name = getattr(backend, "name", None)
    if callable(backend_name):
        backend_name = backend_name()
    if backend_name is None:
        backend_name = str(backend)

    return two_qubit_ops, depth

In [ ]:


from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()  # uses your saved account credentials
count_gates = []
count_depth = []
for n in range(2,21):

    backend = service.backend("ibm_fez")

    ghz_circuit = QuantumCircuit(n)

    ghz_circuit.h(0)

    # Entangle the remaining qubits with the first qubit
    for qubit in range(1, n):
        ghz_circuit.cx(0, qubit)

    # Measure all qubits
    ghz_circuit.measure_all()

    gates, depth = print_circuit_metrics(ghz_circuit, backend, transpile_kwargs={"optimization_level": 1})

    count_depth.append(depth)
    count_gates.append(gates)

for i, val in enumerate(count_gates):
    print(f"({i+2}), ({val})")

for i, val in enumerate(count_depth):
    print(f"({i+2}), ({val})")

